# Candle Streaks 02 — Does causal volatility change directional streak probabilities?

**Historical exploratory.** No fixed candle windows or maximum run length. At every completed five-minute close, combine the current signed streak age with **known trailing volatility level** (24h realized variance) and **known expansion/compression** (1h versus 24h variance). Estimate the full **P(next DOWN / ZERO / UP)** probability matrix using TRAIN only. Compare out-of-sample multinomial forecasts against sign-only, streak-only, sign-vol and additive models, with complete-UTC-day bootstraps. Original AM-T-HAR-Blend is **not modified**. See the frozen [CS02 protocol](../../discussions/Candle_Streaks/002_chatgpt_streak_volatility_probability_matrix_preregistration.md).

A future bar's direction and realized volatility are **never** used to calculate its own predictive inputs. Observations with a current zero-return candle remain outside the scope of this signed-streak analysis; the **next** zero return is scored as a legitimate target.

## 0 — Setup and regression tests

In [ ]:
import os,sys,json,time,subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Run within the cloned repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as vd14
from data_engineering.modeling import candle_streak_volatility as cv
cfg=cv.Specification()
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
assert HIST.is_dir(),f'Missing five-minute Binance price files at {HIST}'
OUT=ROOT/'notebooks'/'Candle_Streaks'/'02';OUT.mkdir(exist_ok=True,parents=True)
pytest_env=os.environ.copy();pytest_env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+pytest_env.get('PYTHONPATH','')
results=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_volatility.py')],cwd=ROOT,env=pytest_env,capture_output=True,text=True)
print(results.stdout[-2500:]);print(results.stderr[-600:]);assert results.returncode==0
T0=time.monotonic();print('Configured assets:',cfg.assets)

## 1 — Load continuous Binance five-minute price history and exclude partial UTC days

The loader uses close-to-close returns; zeros are legitimate returns. We compute volatility from strictly contiguous historical returns, then score only events whose forecast origin **and next bar** lie on verified full UTC days and in the same chronological partition. Note the actual final source date may be earlier/incomplete relative to the configured end.

In [ ]:
raw,excluded=vd14.load_returns(HIST,tuple(cfg.assets),end=pd.Timestamp(cfg.cutoff_utc),min_days=365)
assert {'BTC','ETH'}.issubset(set(raw.underlying.unique())),'BTC/ETH required primary assets'
events,quality=cv.build_vol_events(raw,fast_bars=cfg.fast_bars,slow_bars=cfg.slow_bars)
full_days=cv.complete_utc_days(raw)
raw_end=str(pd.to_datetime(raw.state_time,utc=True).max())
print('Actual last timestamp:',raw_end,'; exclusions:',excluded)
print('Verified complete UTC days:',{a:len(ds) for a,ds in full_days.items()})
assets=sorted(events.underlying.unique());display(quality)
assert len(events)>0

## 2 — TRAIN-only volatility bins; nine level × phase conditions

Level = log(mean past 288 squared five-minute returns). Phase = log(mean past 12 squared returns / mean past 288 squared returns). Both include the just-completed current bar and are available at the decision time. Two cutpoints per feature are frozen on TRAIN only. No fitted AM-T-HAR-Blend forecast is used in this module.

In [ ]:
prepared={};thresholds={};period_rows=[];split_info={}
for asset in assets:
    e=events.loc[events.underlying==asset].copy()
    x,meta=cv.split_eligible(e,full_days[asset],cfg.train_fraction,cfg.valid_fraction)
    bins=cv.fit_vol_bins(x.loc[x.period=='train'])
    x=cv.apply_vol_bins(x,bins)
    prepared[asset]=x;thresholds[asset]=bins;split_info[asset]=meta
    for period,g in x.groupby('period'):
        period_rows.append({'asset':asset,'period':period,'eligible_origin_rows':len(g),
                            'scorable_next_bar_events':int(g.scorable.sum()),
                            'scored_zero_targets':int((g.loc[g.scorable,'next_sign']==0).sum()),
                            'complete_days':int(g.date.nunique())})
period_counts=pd.DataFrame(period_rows);display(period_counts)
print('Bin thresholds:',json.dumps(thresholds,indent=2))

## 3 — Train probability matrices and generate strictly held-out forecasts

Six registered three-class models: IID, sign-only, streak-only, sign+volatility, additive combination, and exact (sign, reached age, level, phase) joint matrix. Every observed streak length is retained. Sparse/unseen cells shrink or back off to preregistered parent forecasts. Scoring includes zero next returns as a third class.

In [ ]:
train_matrix=[];all_descriptive=[];all_scores=[];all_boot=[];all_month=[];fit_summary={};model_fits={}
for asset in assets:
    x=prepared[asset]
    train=x[x.period=='train'];fit=cv.fit_matrix(train,shrink_sign=cfg.shrink_sign,
                      shrink_streak=cfg.shrink_streak,shrink_sign_vol=cfg.shrink_sign_vol,
                      shrink_joint=cfg.shrink_joint)
    model_fits[asset]=fit
    keys=[tuple(z) for z in x[['sign','length','level_bin','trend_bin']].drop_duplicates().itertuples(index=False,name=None)]
    train_matrix.append(cv.probability_matrix(train,fit,include_keys=keys).assign(asset=asset))
    all_descriptive.append(cv.descriptive_matrix(x).assign(asset=asset))
    all_month.append(cv.monthly_stability(x,min_n=50).assign(asset=asset))
    fit_summary[asset]={'n_train_scored':fit['n_train'],'train_matrix_cells':len(fit['counts']['joint']),
                        'max_streak_length_fit':int(train.length.max()),'max_streak_length_all':int(x.length.max()),
                        'unconditional_train_probabilities':fit['global'].tolist()}
    for period in ['valid','test']:
        ev=x[(x.period==period)&x.scorable].copy()
        pred=cv.predict_matrix(ev,fit)
        summary,per_origin=cv.score_threeclass(ev,pred)
        all_scores.append(summary.assign(asset=asset,period=period))
        all_boot.append(cv.day_bootstrap(per_origin,B=cfg.boot,seed=cfg.seed).assign(asset=asset,period=period))
matrix_fit=pd.concat(train_matrix,ignore_index=True)
matrix_descriptive=pd.concat(all_descriptive,ignore_index=True)
monthly=pd.concat(all_month,ignore_index=True)
scores=pd.concat(all_scores,ignore_index=True)
bootstrap=pd.concat(all_boot,ignore_index=True)
display(scores.pivot_table(index=['asset','period'],columns='model',values='logloss').round(7))
print('TRAIN-probability matrix rows:',len(matrix_fit),'Full-history descriptive rows:',len(matrix_descriptive))
display(matrix_fit[(matrix_fit.asset=='BTC')&(matrix_fit.sign==1)&(matrix_fit.length==3)].head(9))

## 4 — Volatility-conditioned reversal/continuation matrix visualizations

Shown for human inspection at selected ages only. The saved probability matrix keeps **all** observed ages, including rare very long streaks. Sample sizes matter; small cells are strongly shrunk toward parent models.

In [ ]:
for asset in assets:
    q=matrix_fit[(matrix_fit.asset==asset)&(matrix_fit.sign==1)&(matrix_fit.length.isin([1,2,3,4,5,6,7,8]))]
    fig,ax=plt.subplots(figsize=(9,4))
    for lvl,phase,label in [(0,0,'Low / compression'),(1,1,'Mid / neutral'),(2,2,'High / expansion')]:
        s=q[(q.level_bin==lvl)&(q.trend_bin==phase)].sort_values('length')
        if len(s):ax.plot(s.length,100*s.p_continue_given_nonzero,marker='o',label=label)
    ax.axhline(50,linestyle='--',color='gray',alpha=.65,label='50%')
    ax.set(title=f'{asset}: TRAIN-fitted probability of upward continuation (given nonzero next bar)',
           xlabel='Current upward streak length',ylabel='Conditional continuation %')
    ax.legend(fontsize=8);plt.show()

## 5 — Paired complete-day bootstrap and volatility-alignment negative control

Primary decision requires the joint matrix to beat **streak-only, sign+vol, and additive** on held-out BTC and ETH log loss, with a negative upper 95% paired-complete-day bootstrap interval for each. Day averaging is reported alongside bar-weighted differences. Volatility-label randomization retains the sign/length cohort, shuffling the bivariate volatility condition jointly within cohort; this is a *diagnostic*, not a fully generated market simulation.

In [ ]:
null_rows=[]
for asset in assets:
    x=prepared[asset];test=x[(x.period=='test')&x.scorable]
    null_rows.append({'asset':asset,**cv.volatility_alignment_null(test,model_fits[asset],reps=100,seed=cfg.seed)})
alignment_null=pd.DataFrame(null_rows)
selected=bootstrap[(bootstrap.period=='test')&(bootstrap.metric=='logloss')]
display(selected[['asset','model','reference','mean_day_difference','mean_bar_weighted_difference',
                  'ci025','ci975','n_days','min_events_day','max_events_day']].round(8))
display(alignment_null)

## 6 — Fixed exploratory decision and compact output files

**No claim of a directional edge from descriptive matrices alone.** A matrix can have striking cells while still failing held-out proper-scoring tests. No AM-T-HAR-Blend or collectors are modified; a later matched-frequency return-density study is required for integration.

In [ ]:
primary=[]
for asset in ('BTC','ETH'):
    bs=bootstrap[(bootstrap.asset==asset)&(bootstrap.period=='test')&(bootstrap.metric=='logloss')&(bootstrap.model=='streak_vol_matrix_3class')]
    items=[]
    for ref in ('streak_3class','sign_vol_3class','additive_3class'):
        row=bs[bs.reference==ref].iloc[0]
        items.append({'reference':ref,'delta':float(row.mean_day_difference),
                      'upper95':float(row.ci975),'passes':bool(row.ci975<0)})
    primary.append({'asset':asset,'comparisons':items,'passes_all':all(r['passes'] for r in items)})
verdict='EXPLORATORY_CANDIDATE_REQUIRES_PROSPECTIVE_CONFIRMATION' if all(x['passes_all'] for x in primary) else 'NO_CONFIRMED_INCREMENTAL_VOLATILITY_CONDITIONING'
decision={'status':'HISTORICAL_EXPLORATORY','verdict':verdict,'primary_assets':primary,
          'AM_T_HAR_Blend_modified':False,'prospective_confirmation':False,
          'note':'Observed trailing 1h/24h RV is NOT an AM-T-HAR-Blend fitted forecast; TEST is historically reused.'}
manifest={'module':'Candle_Streaks_02','specification':cv.registration(),'source':'Binance spot 5m returns via VD14 loader',
          'actual_source_last_timestamp':raw_end,'excluded_assets':excluded,'complete_day_splits':split_info,
          'quality':quality.to_dict('records'),'registered_models':list(cv.MODELS),'train_volatility_bins':thresholds,
          'model_fit_summary':fit_summary,'decision':decision}
quality.to_csv(OUT/'02_quality.csv',index=False)
period_counts.to_csv(OUT/'02_period_counts.csv',index=False)
matrix_fit.to_csv(OUT/'02_probability_matrix_fit.csv',index=False)
matrix_descriptive.to_csv(OUT/'02_probability_matrix_descriptive.csv',index=False)
monthly.to_csv(OUT/'02_monthly_stability.csv',index=False)
scores.to_csv(OUT/'02_oos_direction_scores.csv',index=False)
bootstrap.to_csv(OUT/'02_paired_day_bootstrap.csv',index=False)
alignment_null.to_csv(OUT/'02_volatility_alignment_null.csv',index=False)
(OUT/'02_volatility_bins.json').write_text(json.dumps(thresholds,indent=2))
(OUT/'02_manifest.json').write_text(json.dumps(manifest,indent=2))
(OUT/'02_decision.json').write_text(json.dumps(decision,indent=2))
print('Decision:',verdict)
print('Output files:',sorted(p.name for p in OUT.iterdir() if p.suffix in ('.csv','.json')))
print('Seconds elapsed:',round(time.monotonic()-T0,1))

## Interpretation guardrails

- **The full signed-streak count is unbounded.** Plot age groupings and the 3×3 volatility tertile grid are only descriptive convenience.
- **Predictions are unconditional over next-bar UP, DOWN, ZERO.** The optional `p_continue_given_nonzero` derived from those probabilities is not the primary actionable forecast.
- TRAIN cutoff for volatility bins is frozen; no future realized variance or future sign is a regressor. No fitted AM-T-HAR-Blend parameter is imported or changed.
- TEST day bootstrap excludes incomplete calendar days, correcting CS01's short-final-day issue.
- A probability matrix can be descriptive without improving proper probabilistic scores. BTC and ETH are the primary gate.
- If positive, next research must replace trailing realized-variance proxies with genuine **ex-ante matched-horizon AM-T-HAR-Blend outputs**, recalibrate full signed-return Student-t likelihood, and validate on untouched future periods and options costs. Historical TEST alone is not confirmation.